# Robot omni

## Wstęp

Aby robot mógł dojechać do zadanego punktu, musi znać własne położenie. Żaden z czujników robota nie mierzy go jednak bezpośrednio i bez błędu. Na laboratorium porównamy kilka źródeł informacji o położeniu robota omnikierunkowego jeżdżącego w pudle 1 × 1 m:

* **odometria (odometry)** - całkowanie obrotów kół; estymata jest gładka, ale jej błąd rośnie wraz z przejechaną drogą,
* **LiDAR** - odległości do ścian pudła; nie dryfuje, ale jest zaszumiony,
* **czujniki odległości (distance sensors)**, laserowy ToF i ultradźwiękowy - tanie, ale mierzą odległość tylko w jednym kierunku,
* **filtr Kalmana (Kalman filter)** - łączy gładką odometrię z pomiarem odległości, który nie dryfuje.

Laboratorium składa się z dwóch części. W pierwszej wyprowadzimy kinematykę napędu, wyznaczymy położenie z LiDAR-u i zmierzymy dryf odometrii podczas jazdy i obrotu. W drugiej zaimplementujemy filtr Kalmana w jednym wymiarze i porównamy, jak sprawdza się z LiDAR-em, czujnikiem ToF i czujnikiem ultradźwiękowym.

Wykorzystywane biblioteki: NumPy i pandas do obliczeń, Bokeh do wykresów oraz `omni_drive`, biblioteka sterująca robotem.

**Bezpieczeństwo.** Przed uruchomieniem komórki, która porusza robotem, należy ustawić go na środku pudła, przodem do ściany. Ruch należy zlecać funkcją `await drive_for(...)` (zdefiniowaną niżej), która zatrzymuje robota po zadanym czasie, również w przypadku przerwania komórki (■). Metoda `robot.set_vector(...)` utrzymuje zadany ruch aż do kolejnego polecenia, nawet po zakończeniu komórki. Robota można w każdej chwili zatrzymać wywołaniem `stop()`.

In [ ]:
import asyncio
import math
import time

import numpy as np
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.palettes import Category10_10
from bokeh.plotting import figure
from omni_drive import Angle, DriveVector, OmniDrive

output_notebook(hide_banner=True)

## Połączenie z robotem

Notebook działa na komputerze robota (Raspberry Pi), z którym łączymy się przez sieć Wi-Fi. Gotowość robota sygnalizuje zielona dioda Raspberry Pi migająca w rytmie bicia serca.

`OmniDrive()` nawiązuje połączenie ze sterownikiem kół (ESP32), który 50 razy na sekundę przesyła odometrię i odczyty czujników odległości, oraz z LiDAR-em, który przesyła skan 10 razy na sekundę. `robot.print_status()` wypisuje stan baterii, odometrię, prędkości kół i odczyty czujników. Jeśli komórka zgłosi brak telemetrii, należy odczekać kilka sekund i uruchomić ją ponownie.

Robot ma trzy czujniki ToF (`robot.ir_1`, `ir_2`, `ir_3`: lewy, środkowy, prawy) i dwa ultradźwiękowe (`robot.us_1` lewy, `robot.us_2` środkowy; prawy nie jest podłączony i wyświetla się jako `---`). Na laboratorium korzystamy tylko z czujników środkowych, skierowanych do przodu.

Kierunki ruchu podawane są względem pudła: kąt `0°` oznacza kierunek, w którym robot był zwrócony przodem w chwili wywołania `robot.reset_pose()`. Podczas jazdy bez obrotu biblioteka utrzymuje orientację robota regulatorem, więc robot nie skręca, nawet gdy koła się ślizgają.

In [ ]:
robot = OmniDrive()
await asyncio.sleep(1.0)
robot.reset_pose()
robot.print_status()

### Funkcje pomocnicze

Komórka poniżej definiuje funkcje używane w dalszej części notebooka:

* `stop()` zatrzymuje robota,
* `await drive_for(vector, seconds)` jedzie z zadanym wektorem przez `seconds` sekund i zatrzymuje robota, również po przerwaniu komórki,
* `odometry()` zwraca położenie z odometrii jako tablicę `[x, y, yaw]` (metry i radiany) względem chwili ostatniego `robot.reset_pose()`,
* `reading(sensor)` zwraca odczyt czujnika odległości w metrach albo `nan`, gdy czujnik nie zmierzył odległości,
* `lidar_points()` zwraca punkty ostatniego skanu LiDAR-u jako tablicę N × 2 w metrach, w układzie robota (oś $x$ do przodu, oś $y$ w lewo), bez odbić od obudowy robota,
* `Recorder(nazwa=funkcja, ...)` co 50 ms zapisuje wartości podanych funkcji; `start()` rozpoczyna nagrywanie, a `stop()` je kończy i zwraca tabelę pandas z kolumną `t` (czas w sekundach),
* `plot_series(t, series, title, y_label, dots=())` rysuje przebiegi czasowe ze słownika `series`; serie wymienione w `dots` rysowane są punktami.

Przy poleceniu `velocity = 1.0` obwód koła porusza się z prędkością $s_{max} \approx 0{,}39$ m/s (`S_MAX`), co wynika z parametrów enkoderów: maksymalnie 175 impulsów na 100 ms, 990 impulsów na obrót, promień koła 35 mm.

In [ ]:
S_MAX = 0.39
LIDAR_BODY_RANGE_M = 0.2


def stop():
    robot.set_vector(DriveVector.zero())


async def drive_for(vector, seconds):
    robot.set_vector(vector)
    try:
        await asyncio.sleep(seconds)
    finally:
        stop()
    await asyncio.sleep(0.3)


def odometry():
    pose = robot.pose
    return np.array([pose.x.meters, pose.y.meters, pose.yaw.radians])


def reading(sensor):
    distance = sensor.distance
    if distance is None or not 0 < distance.meters < math.inf:
        return math.nan
    return distance.meters


def lidar_points():
    scan = robot.lidar
    if scan is None or not scan.points:
        return np.empty((0, 2))
    points = np.array(scan.points, dtype=float) / 1000.0
    return points[np.hypot(points[:, 0], points[:, 1]) > LIDAR_BODY_RANGE_M]


class Recorder:
    def __init__(self, **channels):
        self.channels = channels
        self.rows = []

    def start(self, period=0.05):
        self._task = asyncio.create_task(self._record(period))
        return self

    def stop(self):
        self._task.cancel()
        return pd.DataFrame(self.rows)

    async def _record(self, period):
        start = time.monotonic()
        while True:
            row = {"t": time.monotonic() - start}
            row |= {name: read() for name, read in self.channels.items()}
            self.rows.append(row)
            await asyncio.sleep(period)


def plot_series(t, series, title, y_label, dots=()):
    p = figure(title=title, width=820, height=320, x_axis_label="t [s]", y_axis_label=y_label)
    for color, (name, values) in zip(Category10_10, series.items(), strict=False):
        if name in dots:
            p.scatter(t, values, size=4, color=color, legend_label=name)
        else:
            p.line(t, values, line_width=2, color=color, legend_label=name)
    p.legend.click_policy = "hide"
    show(p)

## Część 1: Ruch i odometria

### Kinematyka napędu

Robot ma trzy **koła omnikierunkowe (omni wheels)** rozmieszczone co 120° w odległości $L = 0{,}132$ m od środka. Na obwodzie każdego koła znajdują się bierne rolki. Koło napędza więc robota wyłącznie w kierunku stycznym do okręgu, na którym leżą koła, a w kierunku prostopadłym przetacza się swobodnie na rolkach. Dzięki temu robot może poruszać się w dowolnym kierunku i jednocześnie obracać wokół własnej osi; taki napęd nazywamy **holonomicznym (holonomic)**.

![Koło omnikierunkowe: ruch w kierunku napędu, a następnie swobodny ruch na rolkach. Model koła kiddo_omniwheel (makerspet, Apache 2.0)](img/omni_wheel_rolling.gif)

![Rozmieszczenie kół robota i kierunki, w których każde z nich napędza platformę](img/wheel_forces_top.png)

Koło ustawione pod kątem $\beta_i$ napędza robota w kierunku $(-\sin\beta_i, \cos\beta_i)$, zatem jego obwód porusza się z prędkością

$$
w_i = -\sin(\beta_i)\, v_x + \cos(\beta_i)\, v_y + L\, \omega,
$$

gdzie $(v_x, v_y)$ jest prędkością robota w jego własnym układzie (oś $x$ do przodu, oś $y$ w lewo), a $\omega$ prędkością obrotu. W naszym robocie $\beta_1 = 60°$, $\beta_2 = 180°$, $\beta_3 = 300°$. Trzy równania tworzą układ $\mathbf{w} = \mathbf{M}\mathbf{v}$, gdzie wiersz $i$ macierzy $\mathbf{M}$ to $[-\sin\beta_i,\ \cos\beta_i,\ L]$. Biblioteka korzysta z niego, zamieniając polecenie `set_vector` na prędkości kół.

Odometria działa w odwrotną stronę. Jeżeli od poprzedniego kroku obwody kół przebyły drogi $\Delta s_1, \Delta s_2, \Delta s_3$, to rozwiązanie układu $\mathbf{M}\,[\Delta x, \Delta y, \Delta\theta]^T = [\Delta s_1, \Delta s_2, \Delta s_3]^T$ jest przesunięciem w układzie robota. Do układu pudła przesunięcie obraca się o kąt robota w połowie kroku, $\bar\theta = \theta + \Delta\theta / 2$:

$$
x \leftarrow x + \Delta x \cos\bar\theta - \Delta y \sin\bar\theta, \qquad
y \leftarrow y + \Delta x \sin\bar\theta + \Delta y \cos\bar\theta, \qquad
\theta \leftarrow \theta + \Delta\theta
$$

W ten sam sposób, 100 razy na sekundę, odometrię wyznacza sterownik robota.

**Zadanie 1**

1. Zbuduj macierz `M` (3 × 3) dla kątów zapisanych w `WHEEL_ANGLES`. Funkcje `np.sin` i `np.cos` przyjmują argument w radianach.
2. Uzupełnij funkcję `odometry_step(pose, ds)`. Funkcja przyjmuje **pozę (pose)**, czyli tablicę `[x, y, yaw]`, oraz przesunięcia obwodów kół `ds` (trzy wartości w metrach) i zwraca nową pozę. Przesunięcie w układzie robota wyznacz funkcją [`np.linalg.solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html).

Komórka pod zadaniem sprawdza rozwiązanie na przejeździe 1 m na wprost w krokach po 2 mm. Przy idealnych kołach odometria powinna wskazać $(1, 0, 0°)$. Następnie przejazd jest powtarzany dla koła 1 o 2% większego, niż zakłada model.

In [ ]:
L = 0.132
WHEEL_ANGLES = np.radians([60, 180, 300])

M = ...


def odometry_step(pose, ds): ...

In [ ]:
def drive_straight(wheel_scale):
    pose = np.zeros(3)
    for _ in range(500):
        pose = odometry_step(pose, wheel_scale * (M @ [0.002, 0.0, 0.0]))
    return pose


for name, scale in (("exact wheels", [1.0, 1.0, 1.0]), ("wheel 1 +2%", [1.02, 1.0, 1.0])):
    x, y, yaw = drive_straight(np.array(scale))
    print(f"{name:14s} x = {x:.3f} m, y = {y * 100:+.1f} cm, yaw = {math.degrees(yaw):+.1f}°")

Różnica 2% w promieniu jednego koła powoduje po przejechaniu metra odchylenie o kilka centymetrów w bok i kilka stopni obrotu, i to w modelu pozbawionym poślizgu. Odometria całkuje każdy taki błąd i nie ma mechanizmu, który by go korygował.

### Położenie z LiDAR-u

Drugim źródłem położenia jest LiDAR Hokuyo URG-04LX-UG01, który 10 razy na sekundę mierzy około 680 odległości w polu widzenia 240°, w zakresie do 5,6 m, z dokładnością około 3 cm. Z tyłu robota LiDAR ma martwą strefę.

W pudle położenie można wyznaczyć z odległości do ścian. Regulator orientacji utrzymuje robota przodem do tej samej ściany, więc gdy robot przesunie się o $x$ do przodu, odległość do przedniej ściany zmaleje o $x$, a gdy przesunie się o $y$ w lewo, odległość do lewej ściany zmaleje o $y$, a do prawej wzrośnie o $y$. Położenie względem startu to zatem

$$
x = f_0 - f, \qquad y = \frac{(l_0 - l) - (r_0 - r)}{2},
$$

gdzie $f$, $l$, $r$ to odległości do przedniej, lewej i prawej ściany, a indeks 0 oznacza odległości zmierzone na starcie. Uśrednienie obu ścian bocznych zmniejsza szum. Odległość do ściany w danym kierunku wyznaczymy jako medianę odległości punktów skanu leżących w wąskim wycinku wokół tego kierunku; mediana jest odporna na pojedyncze błędne punkty.

Takie położenie nie dryfuje, ponieważ ściany się nie przesuwają, ale działa tylko w znanym, prostokątnym otoczeniu i przy znanej orientacji. Roboty w dowolnym otoczeniu budują z kolejnych skanów mapę i dopasowują do niej każdy nowy skan (**scan matching**); zasada jest ta sama, ale ściany są wyznaczane z danych.

**Zadanie 2**

1. Uzupełnij funkcję `wall_distances(points)`, która zwraca tablicę `[f, l, r]`: odległości do ściany z przodu (kierunek 0°), z lewej (90°) i z prawej (−90°), każdą jako medianę odległości (`np.hypot`) punktów, których kierunek (`np.arctan2(y, x)`) różni się od danego o mniej niż 5°.
2. Uzupełnij funkcję `lidar_position(walls, start_walls)`, która zwraca położenie `[x, y]` ze wzoru powyżej.
3. Uzupełnij funkcję `drive_square(side, velocity)`, która realizuje przejazd po kwadracie o boku `side` metrów bez obrotu robota: kolejno do przodu, w lewo, do tyłu i w prawo (`Angle.deg(0)`, `90`, `180`, `270`). Użyj `await drive_for(DriveVector(velocity, heading), seconds)`. Przy `velocity = 1.0` robot porusza się z prędkością około `S_MAX` m/s.

Komórka pod zadaniem wykonuje 3 okrążenia kwadratu o boku 20 cm z `velocity = 0.3`, rysuje tor według odometrii i LiDAR-u oraz wypisuje, o ile różnią się na końcu przejazdu. Przed uruchomieniem ustaw robota na środku pudła, przodem do ściany.

In [ ]:
def wall_distances(points): ...


def lidar_position(walls, start_walls): ...


async def drive_square(side, velocity): ...

In [ ]:
robot.reset_pose()
await asyncio.sleep(0.5)
start_walls = wall_distances(lidar_points())


def lidar_xy():
    return lidar_position(wall_distances(lidar_points()), start_walls)


rec = Recorder(
    odom_x=lambda: odometry()[0],
    odom_y=lambda: odometry()[1],
    lidar_x=lambda: lidar_xy()[0],
    lidar_y=lambda: lidar_xy()[1],
).start()
for _ in range(3):
    await drive_square(0.2, 0.3)
await asyncio.sleep(0.5)
square = rec.stop()

p = figure(title="Square, 3 laps", width=420, height=420, match_aspect=True)
p.xaxis.axis_label, p.yaxis.axis_label = "y [m] (left)", "x [m] (forward)"
p.x_range.flipped = True
p.line(square.odom_y, square.odom_x, line_width=2, color="darkorange", legend_label="odometry")
p.line(square.lidar_y, square.lidar_x, line_width=2, color="steelblue", legend_label="LiDAR")
show(p)

end = square.iloc[-1]
gap = math.hypot(end.odom_x - end.lidar_x, end.odom_y - end.lidar_y)
print(f"odometry at the end:  x = {end.odom_x * 100:+.1f} cm, y = {end.odom_y * 100:+.1f} cm")
print(f"LiDAR at the end:     x = {end.lidar_x * 100:+.1f} cm, y = {end.lidar_y * 100:+.1f} cm")
print(f"odometry - LiDAR:     {gap * 100:.1f} cm after {3 * 4 * 0.2:.1f} m")

Robot miał wrócić do punktu startu. Odometria zwykle twierdzi, że wrócił niemal dokładnie, natomiast LiDAR pokazuje, gdzie robot jest naprawdę. Na zamkniętej trasie część błędu odometrii się znosi; podczas jazdy w jednym kierunku błąd byłby większy.

### Obrót

Dotychczas robot poruszał się bez zmiany orientacji. Trzecia składowa `DriveVector`, `omega`, steruje obrotem niezależnie od ruchu postępowego. Wartość `omega = 1.0` oznacza, że wszystkie koła obracają się z pełną prędkością w tym samym kierunku; robot obraca się wtedy z prędkością $\omega_{max} = s_{max} / L$.

![Jazda na wprost, obrót w miejscu i jazda z obrotem. Prędkości kół wynikają z kinematyki odwrotnej](img/kiwi_motion.gif)

Podczas obrotu poślizg kół jest większy niż podczas ruchu postępowego, a odometria tego nie widzi. Kąt robota zmierzymy więc również LiDAR-em: funkcja `wall_angle()` w komórce pod zadaniem dopasowuje prostą do punktów przedniej ściany i zwraca kąt, o jaki robot jest obrócony względem niej.

**Zadanie 3**

1. Wyznacz `OMEGA_MAX`, czyli prędkość obrotu w rad/s przy `omega = 1.0`, na podstawie `S_MAX` i `L`.
2. Uzupełnij funkcję `rotate(turns, omega=0.3)`, która obraca robota w miejscu o `turns` pełnych obrotów (wartość ujemna oznacza obrót w prawo). Czas obrotu wyznacz z `omega` i `OMEGA_MAX`, a ruch zleć funkcją `drive_for` z `DriveVector(0.0, Angle.deg(0), omega)`.

Komórka pod zadaniem obraca robota o jeden pełny obrót w miejscu i porównuje, o ile kąt robota po obrocie różni się od kąta przed obrotem według odometrii i według LiDAR-u. Przed uruchomieniem ustaw robota na środku pudła, przodem do ściany.

In [ ]:
OMEGA_MAX = ...


async def rotate(turns, omega=0.3): ...

In [ ]:
def wall_angle(points):
    ahead = points[(points[:, 0] > 0) & (np.abs(points[:, 1]) < 0.25)]
    slope = np.polyfit(ahead[:, 1], ahead[:, 0], 1)[0]
    return math.atan(slope)


def wrap(angle):
    return (angle + math.pi) % (2 * math.pi) - math.pi


await asyncio.sleep(0.5)
odometry_before, lidar_before = odometry()[2], wall_angle(lidar_points())
await rotate(1)
await asyncio.sleep(0.5)
odometry_after, lidar_after = odometry()[2], wall_angle(lidar_points())
print(f"OMEGA_MAX = {OMEGA_MAX:.2f} rad/s")
print(
    f"odometry residual after a full turn: {math.degrees(wrap(odometry_after - odometry_before)):+.1f}°"
)
print(f"LiDAR residual after a full turn:    {math.degrees(lidar_after - lidar_before):+.1f}°")

Odometria uważa, że robot wykonał niemal dokładnie pełny obrót, a LiDAR pokazuje różnicę wynikającą z poślizgu kół. Po takim obrocie odometria myli się co do kierunku jazdy, więc każdy kolejny przejazd zwiększa błąd położenia.

## Część 2: Czujniki odległości i filtr Kalmana

### Czujniki ToF i ultradźwiękowe

Czujnik **ToF (time of flight)** VL53L1X wysyła impuls światła podczerwonego i mierzy czas powrotu odbitego światła. Mierzy odległość do ok. 4 m w wąskim stożku (ok. 27°), co 50 ms, z dokładnością kilku milimetrów.

Czujnik **ultradźwiękowy (ultrasonic)** HC-SR04 wysyła impuls o częstotliwości 40 kHz i mierzy czas $t$ do powrotu echa. Dźwięk pokonuje drogę do przeszkody dwukrotnie, zatem odległość wynosi $d = c\,t/2$, gdzie $c \approx 343$ m/s. Sterownik robota ogranicza zasięg do 2 m; gdy echo nie wróci, `reading()` zwraca `nan`.

![HC-SR04: impuls 40 kHz biegnie do ściany i wraca, a pin ECHO pozostaje w stanie wysokim przez czas przelotu](img/ultrasonic_echo.gif)

Fala odbita od ściany ustawionej pod kątem może nie powrócić do czujnika; wówczas czujnik nie zwraca odczytu albo zwraca echo od innego obiektu.

![Pomiar czasu przelotu: echo od ściany prostopadłej i od ściany pod kątem](img/ultrasonic.png)

Błędy pomiaru dzielą się na trzy rodzaje, z których każdy wymaga innego postępowania. **Błąd systematyczny (bias)** usuwa się kalibracją, **szum losowy (random noise)** uśrednia filtr, natomiast **pomiary odstające (outliers)** należy wykryć i odrzucić.

![Trzy rodzaje błędów pomiaru](img/sensor_errors.png)

Oba czujniki mierzą odległość od swojej obudowy, a LiDAR od swojego środka. Nagranie poniżej zaczyna się od sekundy postoju, w której zmierzymy różnicę między odczytami czujników a LiDAR-u i dodamy ją do czujników. W ten sposób usuwamy błąd systematyczny i wszystkie trzy źródła mierzą tę samą wielkość: odległość $d$ od LiDAR-u do przedniej ściany.

Komórka cofa robota o 15 cm, stoi przez sekundę, przejeżdża powoli 30 cm do przodu i 30 cm z powrotem, a po nagraniu wraca na środek pudła. Co 50 ms zapisywane są: odległość do ściany z LiDAR-u, odczyty środkowych czujników ultradźwiękowego i ToF oraz położenie $x$ z odometrii. Przed uruchomieniem ustaw robota na środku pudła, przodem do ściany.

In [ ]:
def lidar_front():
    return wall_distances(lidar_points())[0]


await drive_for(DriveVector(0.3, Angle.deg(180)), 0.15 / (0.3 * S_MAX))
robot.reset_pose()
await asyncio.sleep(0.5)
rec = Recorder(
    lidar=lidar_front,
    ultrasonic=lambda: reading(robot.us_2),
    tof=lambda: reading(robot.ir_2),
    odom_x=lambda: odometry()[0],
).start()
await asyncio.sleep(1.2)
await drive_for(DriveVector(0.15, Angle.deg(0)), 0.3 / (0.15 * S_MAX))
await drive_for(DriveVector(0.15, Angle.deg(180)), 0.3 / (0.15 * S_MAX))
await asyncio.sleep(0.5)
data = rec.stop()
await drive_for(DriveVector(0.3, Angle.deg(0)), 0.15 / (0.3 * S_MAX))

t = data["t"].to_numpy()
still = t < 1.0
for name in ("ultrasonic", "tof"):
    data[name] += np.nanmedian(data["lidar"][still] - data[name][still])
truth = data["lidar"].to_numpy()
u = np.diff(data["odom_x"].to_numpy(), prepend=data["odom_x"].iloc[0])
print(f"samples: {len(data)}, distance from {np.nanmax(truth):.2f} m to {np.nanmin(truth):.2f} m")
plot_series(
    t,
    {"LiDAR": truth, "ultrasonic": data["ultrasonic"], "ToF": data["tof"]},
    title="Approach to the wall",
    y_label="distance to the wall [m]",
    dots=("ultrasonic", "ToF"),
)

### Filtr Kalmana w jednym wymiarze

Odometria i czujnik mierzą tę samą wielkość, odległość $d$ od przedniej ściany, lecz ich błędy mają odmienny charakter. Odometria jest gładka, ale dryfuje; czujnik nie dryfuje, ale każdy jego pomiar jest zaszumiony.

Animacja poniżej przedstawia oba źródła na przykładzie dojazdu do ściany. Odometria (kolor pomarańczowy) oddala się od wartości rzeczywistej, pomiary (kolor szary) rozrzucone są wokół niej, a filtr Kalmana (kolor zielony) łączy oba źródła. Pas ±2σ poszerza się w okresie braku pomiarów i zwęża po ich powrocie.

![Filtr Kalmana w 1D: robot jedzie do ściany, odometria dryfuje, czujnik szumi, filtr łączy oba źródła](img/kalman_1d_tracking.gif)

Filtr Kalmana przechowuje **estymatę (estimate)** $\hat d$ oraz jej **wariancję (variance)** $P$, będącą miarą niepewności ($\sqrt P$ to odchylenie standardowe w metrach), i naprzemiennie wykonuje dwa kroki:

* **Predykcja (prediction)**, w każdym kroku. Odometria wskazuje, że robot przemieścił się do przodu o $u$ metrów, zatem ściana jest o $u$ bliżej. Model ruchu nie jest dokładny, więc niepewność rośnie o **szum procesu (process noise)** $Q$:
$$\hat d^- = \hat d - u, \qquad P^- = P + Q$$
* **Korekcja (update)**, po otrzymaniu pomiaru $z$ obarczonego **szumem pomiaru (measurement noise)** o wariancji $R$. **Wzmocnienie Kalmana (Kalman gain)** $K$ określa stopień zaufania do pomiaru, od 0 (pomiar ignorowany) do 1 (pomiar przyjmowany bez zmian):
$$K = \frac{P^-}{P^- + R}, \qquad \hat d = \hat d^- + K\,(z - \hat d^-), \qquad P = (1 - K)\,P^-$$

![Pętla filtru Kalmana](img/kalman_loop.png)

Oba kroki można przedstawić na rozkładach prawdopodobieństwa. Predykcja przesuwa rozkład estymaty i go poszerza, a korekcja mnoży go przez rozkład pomiaru. Wynikowy rozkład leży pomiędzy nimi, bliżej źródła o mniejszej wariancji, i jest węższy od obu.

![Predykcja i korekcja na rozkładach normalnych dla dokładnego i dla silnie zaszumionego pomiaru](img/kalman_gauss.gif)

### Odrzucanie pomiarów odstających

Filtr Kalmana zakłada, że szum pomiaru ma rozkład normalny. Fałszywe echo, np. odczyt 1,5 m przy ścianie oddalonej o 30 cm, filtr przyjąłby tak samo jak każdy inny pomiar i estymata gwałtownie by się zmieniła. Zapobiega temu **bramkowanie (gating)**: przed korekcją sprawdzamy, czy pomiar mieści się tam, gdzie się go spodziewamy. Różnica $y = z - \hat d^-$ między pomiarem a predykcją ma wariancję $S = P^- + R$, będącą sumą niepewności estymaty i pomiaru. Pomiar odrzucamy, jeżeli

$$
|y| > n_\sigma \sqrt{S},
$$

typowo z $n_\sigma = 3$.

![Bramkowanie: pomiary mieszczące się w bramce poprawiają estymatę, fałszywe echo zostaje odrzucone](img/gating.gif)

**Zadanie 4**

1. Uzupełnij funkcję `kalman_1d(u, z, d0, P0, Q, R)`, która przetwarza nagranie filtrem Kalmana. W każdym kroku $k$ wykonaj predykcję z przyrostem odometrii `u[k]`, a jeśli `z[k]` nie jest wartością `nan` (`np.isnan`), również korekcję. Funkcja zwraca trzy tablice: estymaty $\hat d$, ich odchylenia standardowe $\sqrt P$ po każdym kroku oraz informację, czy w danym kroku odrzucono pomiar.
2. Dodaj bramkowanie: jeżeli `n_sigma` nie jest `None`, a pomiar nie mieści się w bramce, pomiń korekcję i zapisz informację o odrzuceniu pomiaru.

Komórka pod zadaniem uruchamia filtr na odczytach czujnika ultradźwiękowego z szumem pomiaru $R = (1\text{ cm})^2$ i szumem procesu $Q = (2\text{ mm})^2$ na krok, bez bramkowania i z bramkowaniem, i wyznacza **pierwiastek błędu średniokwadratowego (root mean squared error, RMSE)** względem LiDAR-u. Następnie zwiększ $R$ dziesięciokrotnie i zaobserwuj, jak zmienia się estymata.

In [ ]:
def kalman_1d(u, z, d0, P0, Q, R, n_sigma=None):
    d, P = d0, P0
    estimates, stds, rejected = [], [], []
    ...
    return np.array(estimates), np.array(stds), np.array(rejected)

In [ ]:
def rmse(values, reference):
    ok = ~np.isnan(values) & ~np.isnan(reference)
    return math.sqrt(np.mean((values[ok] - reference[ok]) ** 2))


Q, R = 0.002**2, 0.01**2
z = data["ultrasonic"].to_numpy()
d0 = float(np.nanmedian(z[still]))
plain, std, _ = kalman_1d(u, z, d0, R, Q, R)
gated, _, rejected = kalman_1d(u, z, d0, R, Q, R, n_sigma=3)
odometry_only = d0 - np.cumsum(u)

plot_series(
    t,
    {
        "ultrasonic": z,
        "odometry": odometry_only,
        "LiDAR": truth,
        "Kalman filter": plain,
        "Kalman filter with gating": gated,
        "-2σ": plain - 2 * std,
        "+2σ": plain + 2 * std,
    },
    title="1D Kalman filter, ultrasonic sensor",
    y_label="distance to the wall [m]",
    dots=("ultrasonic",),
)
for name, values in {
    "odometry": odometry_only,
    "ultrasonic": z,
    "Kalman filter": plain,
    "Kalman filter with gating": gated,
}.items():
    print(f"RMSE {name:26s} {rmse(values, truth) * 100:.1f} cm")
print(f"rejected measurements: {rejected.sum()}")

### Porównanie czujników

Ten sam filtr może korzystać z dowolnego czujnika mierzącego odległość od ściany. Porównamy trzy filtry o wspólnej predykcji z odometrii i różnych pomiarach: z LiDAR-u (kilka tysięcy złotych), z czujnika ToF (kilkadziesiąt złotych) i z czujnika ultradźwiękowego (kilka złotych).

Szum pomiaru $R$ każdego czujnika wyznaczymy z danych: podczas pierwszej sekundy nagrania robot stał, więc rozrzut odczytów w tym czasie to szum czujnika, a jego wariancja (`np.nanvar`) jest oszacowaniem $R$.

**Zadanie 5**

Dla każdego źródła pomiaru (`"lidar"`, `"ultrasonic"`, `"tof"`) pobierz odczyty `data[name]`, wyznacz `R` jako wariancję odczytów z pierwszej sekundy (`still`), nie mniejszą niż `1e-6` (1 mm²), oraz `d0` jako ich medianę, a następnie uruchom `kalman_1d` z tym samym `Q` co wcześniej i `n_sigma=3`. Wynik zapisz w słowniku `results[name]`.

Komórka pod zadaniem rysuje estymaty i wypisuje tabelę z $\sqrt R$, RMSE względem LiDAR-u i liczbą odrzuconych pomiarów. LiDAR jest tu również wzorcem, więc RMSE jego filtru pokazuje jedynie, jak bardzo filtr wygładza pomiar.

In [ ]:
results = {}
...

In [ ]:
plot_series(
    t,
    {"LiDAR (reference)": truth}
    | {f"Kalman, {name}": est for name, (est, _, _) in results.items()},
    title="Kalman filter with LiDAR, ultrasonic and ToF measurements",
    y_label="distance to the wall [m]",
)
summary = pd.DataFrame(
    {
        name: {
            "sensor noise √R [mm]": np.nanstd(data[name][still]) * 1000,
            "RMSE vs LiDAR [mm]": rmse(est, truth) * 1000,
            "rejected": int(rejected.sum()),
        }
        for name, (est, _, rejected) in results.items()
    }
).T
summary.round(1)

Zastanów się, który czujnik wybrałbyś do taniego robota sprzątającego, a który do robota, który musi zatrzymać się dokładnie 5 cm przed przeszkodą. Co stanie się z pomiarem czujnika ToF i ultradźwiękowego, gdy robot obróci się o 30° względem ściany?

### Zatrzymanie przed przeszkodą

Ostatnia komórka wykorzystuje czujnik ToF w jego typowym zastosowaniu. Robot jedzie do przodu, aż czujnik zmierzy odległość mniejszą niż 15 cm. Limit czasu odpowiada 25 cm jazdy, dzięki czemu robot nie dojedzie do ściany nawet w przypadku awarii czujnika, a blok `finally` zatrzymuje robota również po przerwaniu komórki. Przed uruchomieniem ustaw robota na środku pudła, przodem do ściany.

In [ ]:
robot.set_vector(DriveVector(0.2, Angle.deg(0)))
try:
    deadline = time.monotonic() + 0.25 / (0.2 * S_MAX)
    while time.monotonic() < deadline and not reading(robot.ir_2) < 0.15:
        await asyncio.sleep(0.02)
finally:
    stop()
print(f"stopped at a ToF reading of {reading(robot.ir_2) * 100:.1f} cm")